In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr 
import random
import string 
from datetime import datetime, timedelta  


In [8]:
spark = SparkSession.builder\
    .appName("Repartition_vs_Coalesce")\
    .master("spark://172.22.0.2:7077")\
    .getOrCreate()
    

In [9]:
sc = spark.sparkContext

#### DataGeneration 

In [28]:
# function to generate random log entry 

def generate_log_entry() : 
    user_id =''.join(random.choices(string.ascii_lowercase + string.digits , k = 7 )) 
    action = random.choice(["login", "logout", "purchase", "click", "view"])
    item_id =''.join(random.choices(string.ascii_lowercase + string.digits , k = 5 )) 
    timestamp = (datetime.now() - timedelta(seconds=random.randint(0, 2592000))).strftime("%Y-%m-%d  %H:%M: %S")
    return (user_id, action, item_id, timestamp)

# generate synthetic data 
log_entries = [generate_log_entry() for _ in range(1000000)]

# create Dataframe

columns = ["user_id", "action", "item_id", "timestamp"]

log_df = spark.createDataFrame(log_entries,columns)



In [29]:
# show sample data 

log_df.show(10, truncate=False)

+-------+--------+-------+---------------------+
|user_id|action  |item_id|timestamp            |
+-------+--------+-------+---------------------+
|tbjvz94|view    |kzr55  |2026-07-13  20:07: 07|
|7922qkp|purchase|az0es  |2026-07-13  07:22: 17|
|za1jr7k|logout  |wevzx  |2026-08-05  06:13: 06|
|05u2ws8|view    |b7pau  |2026-07-07  17:06: 04|
|6e9z7hy|login   |2h8xd  |2026-07-15  02:18: 58|
|su7dqbj|purchase|2ux6i  |2026-07-15  17:57: 37|
|9qa9h0d|login   |d5eqc  |2026-07-30  08:41: 21|
|o23sjcd|logout  |x8pba  |2026-08-06  11:22: 25|
|owvtdlw|purchase|u854h  |2026-07-12  22:42: 32|
|6oujoqs|purchase|0lu0w  |2026-07-17  08:32: 14|
+-------+--------+-------+---------------------+
only showing top 10 rows



26/08/06 14:13:39 WARN TaskSetManager: Stage 2 contains a task of very large size (3819 KiB). The maximum recommended task size is 1000 KiB.


In [32]:
log_df.write.csv('/opt/spark-data/user_logs', header= True , mode='overwrite')

26/08/06 14:22:50 WARN TaskSetManager: Stage 5 contains a task of very large size (3819 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

#### Repartition Function 

In [40]:
log_rdd = sc.textFile('/opt/spark-data/user_logs')


In [49]:
print(f'the original number of partitions : {log_rdd.getNumPartitions()}')

the original number of partitions : 12


In [50]:
coalesce_rdd = log_rdd.coalesce(13)
repartition_rdd = log_rdd.repartition(13)
print(f'the number of partition after coalescing is : {coalesce_rdd.getNumPartitions()}')
print(f'the number of partition after repartitionning is : {repartition_rdd.getNumPartitions()}')

the number of partition after coalescing is : 12
the number of partition after repartitionning is : 13


26/08/06 16:29:40 ERROR TaskSchedulerImpl: Lost executor 0 on 172.22.0.3: worker lost: Not receiving heartbeat for 60 seconds
26/08/07 10:43:05 ERROR TaskSchedulerImpl: Lost executor 1 on 172.22.0.3: worker lost: Not receiving heartbeat for 60 seconds
